In [2]:
from langgraph.graph import StateGraph,START,END
from langgraph.checkpoint.memory import InMemorySaver
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from typing import TypedDict

In [3]:
load_dotenv()

True

In [4]:
llm=ChatOpenAI(model='gpt-4o')

In [5]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explaination:str

In [6]:
def generate_joke(state:JokeState):
    prompt=PromptTemplate(template="""generate a joke on below topic : {topic}""",
                          input_variables=['topic'])
    chain= prompt | llm
    response=chain.invoke({'topic':state['topic']})
    return {'joke':response.content}
    

In [7]:
def generate_explaination(state:JokeState):
    prompt=PromptTemplate(template="""explain the below shared joke : {joke}""",
                              input_variables=['joke'])
    chain= prompt | llm
    response=chain.invoke({'joke':state['joke']})
    return {'explaination':response.content}

In [25]:
graph=StateGraph(JokeState)

In [26]:
# Add Nodes

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explaination',generate_explaination)

# Add Edges
graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explaination')
graph.add_edge('generate_explaination',END)

In [27]:
checkpointer=InMemorySaver()

In [28]:
workflow=graph.compile(checkpointer=checkpointer)

In [29]:
pizza_config={'configurable':{'thread_id':'1'}}

In [36]:
noodels_config={'configurable':{'thread_id':'2'}}

In [37]:
initial_state={'topic':'noodles'}

In [38]:
final_state=workflow.invoke(initial_state,config=noodels_config)

In [15]:
workflow.get_state(config=pizza_config)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza go to the party?\n\nBecause it knew it would be a-slice time!', 'explaination': 'This joke is a classic example of a pun, which is a form of wordplay that exploits multiple meanings of a term or similar-sounding words for humorous or rhetorical effect.\n\nIn this joke, the humor relies on the phrase "a-slice time." This is a play on the common expression "a nice time," which is something you might expect to have at a party. However, the joke replaces "nice" with "slice," a word related to pizza since pizzas are typically cut into slices. The pun works because "a-slice" sounds similar to "a nice," creating a humorous twist by connecting the pizza theme with the idea of having a good time. The expectation is subverted slightly, creating a light-hearted and playful punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d44b9-92b5-6aeb-8002-511cf7343275'}}, metadata={'sou

In [35]:
list(workflow.get_state_history(config=pizza_config))

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza go to the party?\n\nBecause it couldn't resist a little slice of the action! 🍕", 'explaination': 'This joke plays on the double meaning of the phrase "a slice of the action." In a literal sense, pizza is often sliced to be served. However, "a slice of the action" is also an idiom that means taking part in an exciting activity or getting involved in something fun. \n\nThe punchline is amusing because it combines these two meanings: the pizza, which is typically sliced, is personified as wanting to join in on the fun at the party, making it a clever and punny play on words.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d44c1-c50a-6355-8002-60ac36e8e1e1'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-30T10:24:38.284558+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d44c1-a51f-6395-8001-739

In [40]:
list(workflow.get_state_history(config=noodels_config))

[StateSnapshot(values={'topic': 'noodles', 'joke': 'Why did the noodle get invited to every party?\n\nBecause it was the pasta everyone wanted to be around! 🍝😄', 'explaination': 'This joke is a play on words, making use of a pun. Here\'s the explanation:\n\n1. **Noodle/Pasta**: The joke starts with mentioning "noodle," which is a type of pasta. The humor comes from playing with the word "pasta."\n\n2. **Pun on "Pasta"/"Past-a"**: The punchline is "the pasta everyone wanted to be around!" The joke plays on the sound of the word "pasta," which sounds like "past-a." The phrase "the past-a everyone wanted to be around" is a pun on the expression "the person everyone wanted to be around." \n\n3. **Social Context**: In social settings, saying someone is "the person everyone wants to be around" means they are popular or enjoyable to have at gatherings. In this joke, the noodle is humorously credited as being popular or desirable at parties.\n\nSo, the humor comes from the clever wordplay comb

In [41]:
workflow.invoke(None,config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f1d44c3-057d-6f5c-8000-e3dad68b1d95'}})

{'topic': 'noodles',
 'joke': "Why don't noodles ever get invited to parties?\n\nBecause they're always too tangled up in their own problems!",
 'explaination': 'This joke is a play on words that uses the literal and metaphorical meanings of "tangled." \n\n1. **Literal Meaning**: Noodles are known to get physically tangled or intertwined due to their long, stringy nature. This creates a visual image that is humorous because it\'s easy to imagine noodles as being messy and troublesome in a social setting.\n\n2. **Metaphorical Meaning**: The phrase "tangled up in their own problems" is a common expression used to describe someone who is preoccupied or overwhelmed with personal issues, making them unavailable or unfit for social events like parties.\n\nThe humor stems from the clever way the joke combines these two meanings, suggesting that noodles have their own \'problems\' with tangling, preventing them from participating in the social activity of being invited to parties.'}